# Practical 13 — Word Clouds

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To visualise word frequencies in the review corpus as word clouds, compare how stopword removal (NLTK's list, then the custom domain list from Practical 2) changes what the cloud shows, and test how much information a word cloud can actually carry on a 15-review corpus.

## Theory

A **word cloud** draws each word at a size proportional to its frequency, so the most common words stand out visually. It's a quick way to get a feel for a corpus, but it only ever shows **one thing**: relative word frequency. It says nothing about word order, context, or sentiment.

Two things determine whether a word cloud is useful:
- **Preprocessing.** Without stopword removal, the biggest words in almost any English text are "the", "and", "was" — the cloud is dominated by words that say nothing about the content. Domain-specific filler matters too: in a movie-review corpus, "movie" and "film" can dominate even after a standard stopword list is applied (the custom list from Practical 2 targets exactly this).
- **Frequency spread.** A word cloud communicates only through size *differences*. If almost every word appears once, almost every word is drawn at the same size, and the cloud looks like decoration rather than a summary.

One practical detail: `WordCloud.generate(text)` quietly tokenizes, lowercases, and removes its own default stopwords before drawing anything. To keep control over the preprocessing (and keep it comparable with Practicals 1-2), this practical computes frequencies itself and passes them in with `generate_from_frequencies()`. Step 4 then looks at what the library does when you *don't* take that control.

## Algorithm

1. Build three versions of the corpus with identical cleaning: (A) cleaned only, (B) cleaned + NLTK stopwords removed, (C) cleaned + NLTK and custom domain stopwords removed.
2. Print the top 10 words for each version.
3. Measure how flat each frequency distribution is (how many words appear only once).
4. Draw all three word clouds side by side and save the figure to `images/`.
5. Generate a cloud the "default" way from raw text and inspect what the library did on its own.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd
import nltk
import matplotlib.pyplot as plt

nltk.download("stopwords")

import preprocessing
import tokenizer
import word_clouds as wc

df = pd.read_csv("../datasets/sample_reviews.csv")

nltk_stops = preprocessing.get_nltk_stopwords()
custom_stops = preprocessing.load_custom_stopwords("../datasets/stopwords_custom.txt")

# Three versions of the SAME corpus: cleaning is identical, only stopword removal differs.
tokens_cleaned = df["review"].apply(
    lambda r: tokenizer.regex_word_tokenize(preprocessing.clean_text(r))
).tolist()
tokens_nltk_stops = [preprocessing.remove_stopwords(t, nltk_stops) for t in tokens_cleaned]
tokens_all_stops = [preprocessing.remove_stopwords(t, nltk_stops | custom_stops) for t in tokens_cleaned]

versions = {
    "A: cleaned only": tokens_cleaned,
    "B: + NLTK stopwords out": tokens_nltk_stops,
    "C: + NLTK and custom stopwords out": tokens_all_stops,
}
freqs = {name: wc.word_frequencies(tokens) for name, tokens in versions.items()}
print(f"Loaded {len(df)} reviews")


### Step 1 — Top 10 words in each version

In [ ]:
for name, f in freqs.items():
    print(name)
    print(f.most_common(10))
    print()


### Step 2 — How flat is each frequency distribution?

In [ ]:
for name, f in freqs.items():
    print(f"{name}: {wc.frequency_spread(f)}")


**Look at `share_appearing_once` and `max_count` together. A cloud only conveys information through size differences, so ask yourself how much size variation is actually available to draw in each version.**

### Step 3 — The three clouds, side by side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, f) in zip(axes, freqs.items()):
    cloud = wc.make_wordcloud(f)
    ax.imshow(cloud, interpolation="bilinear")
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
os.makedirs("../images", exist_ok=True)
plt.savefig("../images/wordcloud_comparison.png", dpi=150)
plt.show()


### Step 4 — What does the library do when you hand it raw text?

Here the corpus is passed to `WordCloud.generate()` as one raw string, with no preprocessing from us. Check what ended up in the cloud, and compare it against Version A above (which had no stopword removal at all).

In [ ]:
from wordcloud import WordCloud

raw_text = " ".join(df["review"])
default_cloud = WordCloud(random_state=42).generate(raw_text)

print(f"Number of entries the library chose to draw: {len(default_cloud.words_)}")
print(list(default_cloud.words_)[:20])
print()
for word in ["the", "was", "not", "movie", "film"]:
    print(f"{word!r} present in default cloud? {word in default_cloud.words_}")


---
## Output

*Run every cell above top to bottom, then paste or describe your actual output here — the three top-10 lists, the three frequency-spread dictionaries, what the three clouds looked like (and where the saved image went), and what the default-library cloud contained. Don't fill this in until you've actually run it.*


## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- How did the top-10 lists change from version A to B to C? Which words disappeared at each step, and did removing the custom domain list make the remaining words noticeably more informative?
- Look at the `frequency_spread` numbers. What share of words appear only once in each version, and what is the maximum count? What does that mean for how much size variation the clouds can actually show?
- Looking at the three clouds themselves: did they look meaningfully different from each other, and was any of them actually a useful summary of this corpus?
- In Step 4, what did the library do on its own that you didn't ask it to do? Compare that to version A, and explain why generating from frequencies (as the rest of this practical did) gives you more control.
- Is a word cloud a good tool for a corpus this small? Compare this to Practical 12, where cosine similarity gave an honest answer on small data: is this a technique that fails because of small data, or one whose output is just uninformative at this scale?

*(Write 4-6 sentences here in your own words once you've run the notebook.)*


---
## Viva Prep — Practice Questions

1. **What does a word cloud actually encode, and what does it leave out?**
   It encodes only relative word frequency, via font size. It leaves out word order, context, and sentiment, so it can't tell "good" from "not good".

2. **Why are word clouds usually made after stopword removal?**
   Without it, the most frequent words in any English text are function words like "the" and "and", which dominate the picture while saying nothing about the content.

3. **Why might a domain-specific stopword list be needed on top of a standard one?**
   Because words can be high-frequency and uninformative only within a particular domain. In a movie-review corpus, "movie" and "film" survive a general stopword list but still tell you little about what any one review says.

4. **Why does a word cloud look uninformative when almost every word occurs once?**
   Size is the only visual signal, and it comes from frequency differences. If most words have the same count, they are drawn at nearly the same size, so there is nothing to emphasise.

5. **Why compute frequencies yourself and use `generate_from_frequencies()` instead of passing raw text to `generate()`?**
   `generate()` applies its own tokenization, lowercasing, and default stopword list internally. Computing frequencies yourself keeps the preprocessing visible and under your control, and makes clouds comparable with the rest of the pipeline.
